# Deep Energy-Based Generative Models

**Date**: 2025/02/11

This notebook is based on [Tutorial 8: Deep Energy-Based Generative Models](https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial8/Deep_Energy_Models.ipynb) by the University of Amsterdam.

## 1. Introduction

This seminar provides a comprehensive exploration of energy-based deep learning models, with a particular emphasis on their use as generative models. Additionally, we will examine advanced concepts such as score matching, noise-contrastive estimation, and partition function approximations. Furthermore, we will discuss sophisticated Markov Chain Monte Carlo (MCMC) methods, including Stochastic Gradient Langevin Dynamics (SGLD) and Annealed Importance Sampling (AIS), and their role in training and stabilizing energy-based models.

## 2. Energy Models

In the first part of this seminar, we will review the theory of energy-based models (EBMs). Energy-based models are motivated by **density estimation**. That is, given a dataset with many elements, we want to estimate the probability distribution over the **entire data space**. For example, if we model MNIST images, our goal is a probability distribution $q_\theta(\mathbf{x})$ over all possible 28×28 images (pixel intensities). 

Here:
- $p(\mathbf{x})$ denotes the **true data distribution**. We do not know $p(\mathbf{x})$ explicitly, but we have samples from it in the form of our dataset.
- $q_{\theta}(\mathbf{x})$ denotes the **model distribution** we are trying to learn. It is defined through an *unnormalized* density $\exp\bigl(-E_\theta(\mathbf{x})\bigr)$ and a partition function $Z_\theta$ ensuring that the total integral/sum is 1.

### 2.1 Energy Function and Partition Function

**Recap**: We define an energy function $E_\theta(\mathbf{x}) \in \mathbb{R}$ whose scalar output indicates how "unlikely" an input $\mathbf{x}$ is under the model. Crucially, we typically **represent $E_\theta(\mathbf{x})$ with a neural network**, where $\theta$ are its parameters. For any $\mathbf{x}$:

$$
E_\theta(\mathbf{x}) 
\;=\; \text{(Neural Network)}_{\theta}\bigl(\mathbf{x}\bigr).
$$

Data points with low energy are assigned high probability, and vice versa. The energy function defines a probability distribution via the Boltzmann distribution:

$$
q_{\theta}(\mathbf{x}) \;=\; \frac{\exp\bigl(-E_{\theta}(\mathbf{x})\bigr)}{Z_{\theta}}, 
\quad
Z_{\theta} \;=\; \int_{\mathbf{x}} \exp\bigl(-E_{\theta}(\mathbf{x})\bigr)\, d\mathbf{x},
$$

where $Z_{\theta}$ is the **partition function**—an integral or sum over all $\mathbf{x}$. 

> **Note**: For $q_{\theta}(\mathbf{x})$ to be a valid probability distribution, the term $\exp\bigl(-E_{\theta}(\mathbf{x})\bigr)$ must be *integrable* over the data space, ensuring $Z_{\theta}$ is finite.

In high-dimensional problems, we cannot compute $Z_\theta$ in closed form, so we rely on approximate training methods.

### 2.2 Connection to Score Matching

The *score* of a probability distribution $p(\mathbf{x})$ is the gradient of its log-density w.r.t. $\mathbf{x}$:

$$
\boldsymbol{\psi}(\mathbf{x}) = 
\nabla_{\mathbf{x}} \log p(\mathbf{x})
$$

If $p(\mathbf{x}) \propto \exp\bigl(-E_{\theta}(\mathbf{x})\bigr)$, then

$$
\boldsymbol{\psi}(\mathbf{x}, \theta) \;=\; - \nabla_{\mathbf{x}} E_{\theta}(\mathbf{x}).
$$

The idea behind *score matching* (Hyvärinen, 2005) is to learn $E_{\theta}$ by matching this model score to the *true* data score. That is, if $p_{\text{data}}(\mathbf{x})$ is the real distribution, score matching tries to make

$$
\underbrace{- \nabla_{\mathbf{x}} E_{\theta}(\mathbf{x})}_{\text{$\boldsymbol{\psi}(\mathbf{x}, \theta)$}}
\;\approx\;
\underbrace{\nabla_{\mathbf{x}} \log p_{\text{data}}(\mathbf{x})}_{\text{$\boldsymbol{\psi}_\mathbf{x}(\mathbf{x})$}}
$$

everywhere. In practice, we cannot compute $\nabla_{\mathbf{x}} \log p_{\text{data}}(\mathbf{x})$ directly since $p_{\text{data}}$ is unknown. However, several variants of score matching exist that circumvent the intractable partition function. For example:

- **Denoising Score Matching (Vincent, 2011)**: Instead of matching the score directly, one corrupts training data with noise and trains a neural network to predict the *clean* data from the noisy version. This procedure can be shown to learn the gradient of the log-density without requiring explicit normalization.
- **Sliced Score Matching**: Reduces computational costs by projecting data onto random directions and matching scores along those slices.

Score matching’s advantage is that it avoids computing or approximating $Z_{\theta}$. However, it typically requires careful design of the objective and can involve second derivatives if done in its original form. Nevertheless, recent *diffusion-based generative models* (Song and Ermon, 2019) can be seen as a sophisticated application of score matching to learn high-dimensional data distributions.

### 2.3 Noise-Contrastive Estimation

Another popular approach to avoid computing the partition function is *Noise-Contrastive Estimation (NCE)* (Gutmann & Hyvärinen, 2010). NCE transforms the unsupervised density estimation problem into a *binary classification* problem, where the goal is to distinguish:
1. **Real data** samples $\mathbf{x} \sim p(\mathbf{x})$
2. **Noise** samples $\mathbf{y} \sim q_0(\mathbf{y})$, where $q_0(\mathbf{y})$ is a chosen noise distribution (often something simple, such as a Gaussian or a uniform distribution)

We construct a classifier that outputs the probability that a sample comes from $p(\mathbf{x})$ rather than $q_0(\mathbf{y})$. If we let

$$
r_{\theta}(\mathbf{x}) \;=\; \frac{\exp\bigl(-E_{\theta}(\mathbf{x})\bigr)}{q_0(\mathbf{x})},
$$

the NCE objective encourages $r_{\theta}(\mathbf{x})$ to be large for real data and small for noise, leading (under mild conditions) to a consistent estimate of the unnormalized density $\exp\bigl(-E_{\theta}(\mathbf{x})\bigr)$. In practice:

- We sample a mini-batch of real data from $p(\mathbf{x})$ and a mini-batch of noise from $q_0(\mathbf{y})$.
- We train a logistic regression classifier to distinguish the real samples from the noise.
- The ratio $r_{\theta}$ between model probability and noise probability is adjusted to correctly separate real data from noise.

A key challenge is choosing a suitable noise distribution $q_0(\mathbf{y})$—it must overlap sufficiently with the true data distribution for NCE to be effective. In high-dimensional image tasks, we often employ strategies like *annealing* (gradually refining the noise distribution) or using replay buffers of previously sampled points. NCE can be efficient but typically depends heavily on the quality of the noise.

### 2.4 Contrastive Divergence

We focus here on **Contrastive Divergence (CD) (Hinton, 2002)**, which avoids explicitly computing $Z_\theta$. To see where the update rule comes from, consider **maximum likelihood estimation** (MLE). We want to adjust $\theta$ to maximize the log-likelihood of the data under our model $q_\theta$. Equivalently, we **minimize** the negative log-likelihood:

$$
\mathcal{L}_{\mathrm{MLE}}(\theta) 
\;=\; -\, \mathbb{E}_{p(\mathbf{x})}\bigl[\log q_\theta(\mathbf{x})\bigr].
$$

Taking the gradient w.r.t. $\theta$,

$$
\nabla_\theta \,\mathcal{L}_{\mathrm{MLE}}(\theta)
\;=\;
-\, \mathbb{E}_{p(\mathbf{x})}\bigl[\nabla_\theta \log q_\theta(\mathbf{x})\bigr].
$$

Because 
$\log q_\theta(\mathbf{x}) \;=\; -\,E_\theta(\mathbf{x}) - \log Z_\theta,$
we get

$$
\nabla_\theta \log q_\theta(\mathbf{x})
\;=\;
-\,\nabla_\theta E_\theta(\mathbf{x})
\;-\; \nabla_\theta \log Z_\theta.
$$

The key difficulty is $\nabla_\theta \log Z_\theta$. If

$$
Z_\theta 
\;=\; 
\int_{\mathbf{x}} \exp\bigl(-E_\theta(\mathbf{x})\bigr)\, d\mathbf{x},
$$

then

$$
\nabla_\theta \log Z_\theta
\;=\;
\frac{1}{Z_\theta}\,\nabla_\theta Z_\theta
\;=\;
\frac{1}{Z_\theta}
\int_{\mathbf{x}} \nabla_\theta\Bigl[\exp\bigl(-E_\theta(\mathbf{x})\bigr)\Bigr]
\,d\mathbf{x}.
$$

Notice that 
$\exp\bigl(-E_\theta(\mathbf{x})\bigr) / Z_\theta \;=\; q_\theta(\mathbf{x})$. 
Hence,

$$
\nabla_\theta \log Z_\theta
\;=\;
\int_{\mathbf{x}} q_\theta(\mathbf{x})\,
\Bigl(-\,\nabla_\theta E_\theta(\mathbf{x})\Bigr)\, d\mathbf{x}
\;=\;
-\,\mathbb{E}_{q_\theta(\mathbf{x})}\bigl[\nabla_\theta E_\theta(\mathbf{x})\bigr].
$$

Putting it all together, we have

$$
\nabla_\theta \,\mathcal{L}_{\mathrm{MLE}}(\theta)
\;=\;
-\,\mathbb{E}_{p(\mathbf{x})}\bigl[\nabla_\theta \log q_\theta(\mathbf{x})\bigr]
\;=\;
\mathbb{E}_{p(\mathbf{x})}\bigl[\nabla_\theta E_\theta(\mathbf{x})\bigr]
\;-\;
\mathbb{E}_{q_\theta(\mathbf{x})}\bigl[\nabla_\theta E_\theta(\mathbf{x})\bigr].
$$

This is the **update rule** we often see in energy-based modeling:

$$
\nabla_{\theta} \mathcal{L}_{\mathrm{MLE}}(\theta) 
\;=\;
\mathbb{E}_{p(\mathbf{x})}\bigl[\nabla_{\theta} E_{\theta}(\mathbf{x})\bigr] 
\;-\; 
\mathbb{E}_{q_{\theta}(\mathbf{x})}\bigl[\nabla_{\theta} E_{\theta}(\mathbf{x})\bigr].
$$

In practice, we cannot sample directly from $q_{\theta}(\mathbf{x})$, so **Contrastive Divergence** approximates $\mathbb{E}_{q_{\theta}(\mathbf{x})}$ by drawing MCMC samples from the **current** model (for instance, via Langevin dynamics). This yields:

$$
\nabla_{\theta} \mathcal{L}_{\mathrm{MLE}}(\theta) 
\,\approx\, 
\mathbb{E}_{p(\mathbf{x})}\bigl[\nabla_{\theta} E_{\theta}(\mathbf{x})\bigr] 
\;-\; 
\underbrace{\mathbb{E}_{q_{\theta}(\mathbf{x})}\bigl[\nabla_{\theta} E_{\theta}(\mathbf{x})\bigr]}_{\text{approx via MCMC}}.
$$

### 2.5 Sampling from Energy-Based Models

To **draw samples** from $q_\theta(\mathbf{x})$, we typically rely on **Markov Chain Monte Carlo (MCMC)**. One common strategy is **Langevin Dynamics**, which repeatedly:

1. **Drifts** the sample $\mathbf{x}$ along $-\,\nabla_\mathbf{x} E_\theta(\mathbf{x})$.  
2. **Diffuses** by adding Gaussian noise $\omega\sim \mathcal{N}(0,\sigma)$.  

Formally, if $\{\mathbf{x}^0, \mathbf{x}^1, \dots, \mathbf{x}^K\}$ is a Markov chain of length $K$, we start from random noise $\mathbf{x}^0$ (e.g., a Gaussian) and iterate:

$$
\mathbf{x}^k 
\;\leftarrow\; 
\mathbf{x}^{k-1} 
\;-\; \eta\, \nabla_\mathbf{x} E_\theta(\mathbf{x}^{k-1}) 
\;+\; \omega,
$$

where $\eta$ is the step size. If $\eta$ is small and $K$ is large, $\mathbf{x}^K$ approximates a draw from $q_\theta$. In contrastive divergence training, we often keep $K$ fairly small (e.g. 10–60 steps) for computational reasons. Despite being an approximation, it usually suffices to get stable training updates.


In [ ]:
## Standard libraries
import os
import numpy as np
import random

## Imports for plotting
import matplotlib
import matplotlib.pyplot as plt
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg', 'pdf')  # For export
matplotlib.rcParams['lines.linewidth'] = 2.0
import seaborn as sns
sns.reset_orig()

## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
import torch.optim as optim

# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms

# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

# Path to the folder where the datasets are/should be downloaded (e.g. CIFAR10)
DATASET_PATH = "../data"
# Path to the folder where the pretrained models are saved
CHECKPOINT_PATH = "models/"

# Setting the seed
pl.seed_everything(42)

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device: str = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
device = torch.device(device)
print("Device:", device)

We also have pre-trained models that we download below.

In [ ]:
import urllib.request
from urllib.error import HTTPError
# Github URL where saved models are stored for this tutorial
base_url = "https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial8/"
# Files to download
pretrained_files = ["MNIST.ckpt", "tensorboards/events.out.tfevents.MNIST"]

# Create checkpoint path if it doesn't exist yet
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

# For each file, check whether it already exists. If not, try downloading it.
for file_name in pretrained_files:
    file_path = os.path.join(CHECKPOINT_PATH, file_name)
    if "/" in file_name:
        os.makedirs(file_path.rsplit("/",1)[0], exist_ok=True)
    if not os.path.isfile(file_path):
        file_url = base_url + file_name
        print(f"Downloading {file_url}...")
        try:
            urllib.request.urlretrieve(file_url, file_path)
        except HTTPError as e:
            print("Something went wrong. Please try to download the file from the GDrive folder, or contact the author with the full output including the following error:\n", e)

### 2.5 Applications of Energy-based models beyond generation

Modeling the probability distribution for sampling new data is not the only application of energy-based models. Any application which requires us to compare two elements is much simpler to learn because we just need to go for the higher energy. A couple of examples are shown below. A classification setup like object recognition or sequence labeling can be considered as an energy-based task as we just need to find the $Y$ input that minimizes the output $E(X, Y)$ (hence maximizes probability). Similarly, a popular application of energy-based models is denoising of images. Given an image $X$ with a lot of noise, we try to minimize the energy by finding the true input image $Y$.


| ![energy_models_application.png](imgs/energy_models_application.png) | 
|:--:| 
| *[Source](https://deepgenerativemodels.github.io/assets/slides/cs236_lecture11.pdf)* |

## 3. Image generation

As an example for energy-based models, we will train a model on image generation. Specifically, we will look at how we can generate MNIST digits with a very simple CNN model. However, it should be noted that energy models are not easy to train and often diverge if the hyperparameters are not well tuned. We will rely on training tricks proposed in the paper [Implicit Generation and Generalization in Energy-Based Models](https://arxiv.org/abs/1903.08689) by Yilun Du and Igor Mordatch ([blog](https://openai.com/blog/energy-based-models/)). The important part of this notebook is however to see how the theory above can actually be used in a model.

### 3.1 Dataset

First, we can load the MNIST dataset below. Note that we need to normalize the images between -1 and 1 instead of mean 0 and std 1 because during sampling, we have to limit the input space. Scaling between -1 and 1 makes it easier to implement it.

In [ ]:
# Transformations applied on each image => make them a tensor and normalize between -1 and 1
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))
                               ])

# Loading the training dataset. We need to split it into a training and validation part
train_set = MNIST(root=DATASET_PATH, train=True, transform=transform, download=True)

# Loading the test set
test_set = MNIST(root=DATASET_PATH, train=False, transform=transform, download=True)

# We define a set of data loaders that we can use for various purposes later.
# Note that for actually training a model, we will use different data loaders
# with a lower batch size.
train_loader = data.DataLoader(train_set, batch_size=128, shuffle=True,  drop_last=True,  num_workers=4, pin_memory=True)
test_loader  = data.DataLoader(test_set,  batch_size=256, shuffle=False, drop_last=False, num_workers=4)

### 3.2 CNN Model

First, we implement our CNN model. The MNIST images are of size 28x28, hence we only need a small model. As an example, we will apply several convolutions with stride 2 that downscale the images. If you are interested, you can also use a deeper model such as a small ResNet, but for simplicity, we will stick with the tiny network.

It is a good practice to use a smooth activation function like Swish instead of ReLU in the energy model. This is because we will rely on the gradients we get back with respect to the input image, which should not be sparse.

In [ ]:
class Swish(nn.Module):

    def forward(self, x):
        return x * torch.sigmoid(x)


class CNNModel(nn.Module):

    def __init__(self, hidden_features=32, out_dim=1, **kwargs):
        super().__init__()
        # We increase the hidden dimension over layers. Here pre-calculated for simplicity.
        c_hid1 = hidden_features//2
        c_hid2 = hidden_features
        c_hid3 = hidden_features*2

        # Series of convolutions and Swish activation functions
        self.cnn_layers = nn.Sequential(
                nn.Conv2d(1, c_hid1, kernel_size=5, stride=2, padding=4), # [16x16] - Larger padding to get 32x32 image
                Swish(),
                nn.Conv2d(c_hid1, c_hid2, kernel_size=3, stride=2, padding=1), #  [8x8]
                Swish(),
                nn.Conv2d(c_hid2, c_hid3, kernel_size=3, stride=2, padding=1), # [4x4]
                Swish(),
                nn.Conv2d(c_hid3, c_hid3, kernel_size=3, stride=2, padding=1), # [2x2]
                Swish(),
                nn.Flatten(),
                nn.Linear(c_hid3*4, c_hid3),
                Swish(),
                nn.Linear(c_hid3, out_dim)
        )

    def forward(self, x):
        x = self.cnn_layers(x).squeeze(dim=-1)
        return x

In the rest of the notebook, the output of the model will actually not represent $E_{\theta}(\mathbf{x})$, but $-E_{\theta}(\mathbf{x})$. This is a standard implementation practice for energy-based models, as some people also write the energy probability density as $q_{\theta}(\mathbf{x}) = \frac{\exp\left(f_{\theta}(\mathbf{x})\right)}{Z_{\theta}}$. In that case, the model would actually represent $f_{\theta}(\mathbf{x})$. In the training loss etc., we need to be careful to not switch up the signs.

### 3.3 Sampling Buffer

Modern energy-based models (EBMs) for images often rely on **Markov Chain Monte Carlo (MCMC)** methods (e.g., Langevin Dynamics) to sample “fake” data $\mathbf{x}^{-}$ from the model distribution 
$$
q_\theta(\mathbf{x}) 
\;=\; 
\frac{\exp\bigl(-E_\theta(\mathbf{x})\bigr)}{Z_\theta}.
$$
However, **high-dimensional** domains (like images) often require many MCMC steps to reach plausible samples. A **sampling buffer** can help reduce computational overhead by storing MCMC end-states (samples) between training iterations—so we do **not** always start each chain from scratch. 

#### 3.3.1 The Buffer Mechanism

1. **Initialization**  
   We keep an empty buffer $\mathcal{B}$ at the start. Then, each training iteration, we sample some fraction (e.g., 95%) of our initial “fake” images $\mathbf{x}_i^0$ from $\mathcal{B}$ (the last few batches’ final MCMC states). The remaining 5% we initialize from **uniform** or **Gaussian** noise in $[-1,1]$. This ensures **diversity** and prevents the chain from collapsing to previously found modes exclusively.

2. **MCMC Refinement**  
   Let $\mathbf{x}_i^{0}$ be our initial states from $\mathcal{B}$ or random noise. We run $K$ steps of Langevin Dynamics:
   $$
   \mathbf{x}_i^{k} 
   \;\leftarrow\; 
   \mathbf{x}_i^{k-1} 
   \;-\; \eta\,\nabla_{\mathbf{x}} E_\theta(\mathbf{x}_i^{k-1})
   \;+\; \omega,
   \quad 
   \omega \sim \mathcal{N}\bigl(\mathbf{0},\,\sigma I\bigr),
   $$
   where $\eta$ is the “step_size” and $\sigma$ the noise scale. After $K$ iterations, $\mathbf{x}_i^{K}$ is deemed our “fake” sample $\mathbf{x}^-_i$. Algorithmically, we often clamp pixel values to $[-1,1]$ after each step to remain in the valid input range.

3. **Buffer Update**  
   The final states $\mathbf{x}_i^{-}$ become new entries of $\mathcal{B}$. Over time, $\mathcal{B}$ amasses a variety of samples from different training phases, often providing a more accurate starting point for the next iteration’s MCMC updates. If $\mathcal{B}$ exceeds some size (e.g. 8192), we pop the oldest samples.

#### 3.3.2 Why This Helps

- **Faster Convergence of MCMC**:  
  Re-initializing from noise each time may require many gradient steps for $\mathbf{x}^k$ to reach a region of significant probability. By reusing previously converged samples $\mathbf{x}^- \in \mathcal{B}$, we drastically reduce the required MCMC steps to “refresh” them.  
- **Reduced Computational Cost**:  
  Especially in high dimensions, long-run MCMC is costly. The buffer allows short-run MCMC (e.g., 60 steps) to suffice for generating plausible samples.  
- **Sample Diversity**:  
  Injecting a small fraction of brand-new noise (5%) ensures we explore fresh modes and avoid the buffer degenerating into a narrow subset of the distribution.

In [ ]:
class Sampler:

    def __init__(self, model, img_shape, sample_size, max_len=8192):
        """
        Inputs:
            model - Neural network to use for modeling E_theta
            img_shape - Shape of the images to model
            sample_size - Batch size of the samples
            max_len - Maximum number of data points to keep in the buffer
        """
        super().__init__()
        self.model = model
        self.img_shape = img_shape
        self.sample_size = sample_size
        self.max_len = max_len
        self.examples = [(torch.rand((1,)+img_shape)*2-1) for _ in range(self.sample_size)]

    def sample_new_exmps(self, steps=60, step_size=10):
        """
        Function for getting a new batch of "fake" images.
        Inputs:
            steps - Number of iterations in the MCMC algorithm
            step_size - Learning rate nu in the algorithm above
        """
        # Choose 95% of the batch from the buffer, 5% generate from scratch
        n_new = np.random.binomial(self.sample_size, 0.05)
        rand_imgs = torch.rand((n_new,) + self.img_shape) * 2 - 1
        old_imgs = torch.cat(random.choices(self.examples, k=self.sample_size-n_new), dim=0)
        inp_imgs = torch.cat([rand_imgs, old_imgs], dim=0).detach().to(device)

        # Perform MCMC sampling
        inp_imgs = Sampler.generate_samples(self.model, inp_imgs, steps=steps, step_size=step_size)

        # Add new images to the buffer and remove old ones if needed
        self.examples = list(inp_imgs.to(torch.device("cpu")).chunk(self.sample_size, dim=0)) + self.examples
        self.examples = self.examples[:self.max_len]
        return inp_imgs

    @staticmethod
    def generate_samples(model, inp_imgs, steps=60, step_size=10, return_img_per_step=False):
        """
        Function for sampling images for a given model.
        Inputs:
            model - Neural network to use for modeling E_theta
            inp_imgs - Images to start from for sampling. If you want to generate new images, enter noise between -1 and 1.
            steps - Number of iterations in the MCMC algorithm.
            step_size - Learning rate nu in the algorithm above
            return_img_per_step - If True, we return the sample at every iteration of the MCMC
        """
        # Before MCMC: set model parameters to "required_grad=False"
        # because we are only interested in the gradients of the input.
        is_training = model.training
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
        inp_imgs.requires_grad = True

        # Enable gradient calculation if not already the case
        had_gradients_enabled = torch.is_grad_enabled()
        torch.set_grad_enabled(True)

        # We use a buffer tensor in which we generate noise each loop iteration.
        # More efficient than creating a new tensor every iteration.
        noise = torch.randn(inp_imgs.shape, device=inp_imgs.device)

        # List for storing generations at each step (for later analysis)
        imgs_per_step = []

        # Loop over K (steps)
        for _ in range(steps):
            # Part 1: Add noise to the input.
            noise.normal_(0, 0.005)
            inp_imgs.data.add_(noise.data)
            inp_imgs.data.clamp_(min=-1.0, max=1.0)

            # Part 2: calculate gradients for the current input.
            out_imgs = -model(inp_imgs)
            out_imgs.sum().backward()
            inp_imgs.grad.data.clamp_(-0.03, 0.03) # For stabilizing and preventing too high gradients

            # Apply gradients to our current samples
            inp_imgs.data.add_(-step_size * inp_imgs.grad.data)
            inp_imgs.grad.detach_()
            if inp_imgs.grad is not None:
                inp_imgs.grad.zero_()
            inp_imgs.data.clamp_(min=-1.0, max=1.0)
            # TODO: a soft clipping function inp_imgs.data.tanh_()

            if return_img_per_step:
                imgs_per_step.append(inp_imgs.clone().detach())

        # Reactivate gradients for parameters for training
        for p in model.parameters():
            p.requires_grad = True
        model.train(is_training)

        # Reset gradient calculation to setting before this function
        torch.set_grad_enabled(had_gradients_enabled)

        if return_img_per_step:
            return torch.stack(imgs_per_step, dim=0)
        else:
            return inp_imgs

### 3.4 Training

With the sampling buffer being ready, we can complete our training algorithm. Below is shown a summary of the full training algorithm of an energy model on image modeling:

<center width="100%" style="padding: 15px"><img src="imgs/training_algorithm.svg" width="700px"></center>

#### 3.4.1 The “Offset” Problem
Because only **relative** energy values (e.g., $E_\theta(\mathbf{x}^+) - E_\theta(\mathbf{x}^-)$) matter for likelihood, the model can shift $E_\theta(\cdot)$ by an **arbitrary constant** $c$:

$$
E_\theta(\mathbf{x}) \;\rightarrow\; E_\theta(\mathbf{x}) + c
$$

without changing the overall probability assignments $\exp(-E_\theta)/Z_\theta$. Indeed, if all energies are shifted by $+c$, the partition function $\exp(-c)\,Z_\theta$ is likewise scaled, leaving $q_\theta(\mathbf{x})$ identical. In practice, this means the EBM is **not constrained** to keep energies in any particular numeric range: it can let them drift arbitrarily large or negative, which can cause numerical instability or training divergence.

#### 3.4.2 L2 Regularization on $E_\theta$

To mitigate unbounded energy drifts, many works add a small **L2 penalty** on the network’s energy outputs. Concretely, a **regularization** term:

$$
\mathcal{L}_{\mathrm{RG}} \;=\; \frac{1}{N}\sum_{i=1}^N \Bigl(E_\theta(\mathbf{x}_i^+)\Bigr)^2 \;+\; \Bigl(E_\theta(\mathbf{x}_i^-)\Bigr)^2
$$

ensures the absolute value of $E_\theta$ for both real and fake samples is “encouraged” to stay near zero. We weight it by $\alpha \ll 1$, yielding a total loss of

$$
\mathcal{L}_{\mathrm{total}}
\;=\;
\mathcal{L}_{\mathrm{CD}}
\;+\;
\alpha \,\mathcal{L}_{\mathrm{RG}},
$$
where $\mathcal{L}_{\mathrm{CD}}$ is the **contrastive divergence** term.

#### 3.4.3 Practical Tips

1. **Balancing $\eta$ and $\sigma$**:  
   If $\eta$ (the step size) is too large or $\sigma$ is too small, MCMC might collapse to a few modes or become unstable. Tuning these hyperparameters is vital.

2. **Regularization $\alpha$**:  
   Since the raw contrastive divergence does **not** constrain the magnitude of $E_\theta(\mathbf{x})$, we add a small penalty $\alpha\,\mathcal{L}_{\mathrm{RG}}$ to keep energies near zero. This prevents unbounded drift in the bias term of the network. A typical $\alpha$ might be $0.1$ or $0.01$, but it must be tuned per dataset/model.

3. **Short-Run vs. Long-Run MCMC**:  
   - If $K$ is small (like 10–60 steps), we rely heavily on a **good buffer** initialization.  
   - If you attempt *long-run MCMC*, you get higher-quality negative samples but with significantly higher computational cost.

4. **Buffer Size**:  
   Keeping $\mathcal{B}$ large (e.g., 8k) helps store diverse states. If it is too small, you might over-exploit a narrow region of $\mathbf{x}$-space.


Below, we put this training dynamic into a PyTorch Lightning module. Remember that, since we model $f_{\theta}(x)=-E_{\theta}(x)$, we need to be careful with switching all important signs, e.g. in the loss function.

In [ ]:
class DeepEnergyModel(pl.LightningModule):

    def __init__(self, img_shape, batch_size, alpha=0.1, lr=1e-4, beta1=0.0, **CNN_args):
        super().__init__()
        self.save_hyperparameters()

        self.cnn = CNNModel(**CNN_args)
        self.sampler = Sampler(self.cnn, img_shape=img_shape, sample_size=batch_size)
        self.example_input_array = torch.zeros(1, *img_shape)

    def forward(self, x):
        z = self.cnn(x)
        return z

    def configure_optimizers(self):
        # Energy models can have issues with momentum as the loss surfaces changes with its parameters.
        # Hence, we set it to 0 by default.
        optimizer = optim.Adam(self.parameters(), lr=self.hparams.lr, betas=(self.hparams.beta1, 0.999))
        scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.97) # Exponential decay over epochs
        return [optimizer], [scheduler]

    def training_step(self, batch, batch_idx):
        # We add minimal noise to the original images to prevent the model from focusing on purely "clean" inputs
        real_imgs, _ = batch
        small_noise = torch.randn_like(real_imgs) * 0.005
        real_imgs.add_(small_noise).clamp_(min=-1.0, max=1.0)

        # Obtain samples
        fake_imgs = self.sampler.sample_new_exmps(steps=60, step_size=10)

        # Predict energy score for all images
        inp_imgs = torch.cat([real_imgs, fake_imgs], dim=0)
        real_out, fake_out = self.cnn(inp_imgs).chunk(2, dim=0)

        # Calculate losses
        reg_loss = self.hparams.alpha * (real_out ** 2 + fake_out ** 2).mean()
        cdiv_loss = fake_out.mean() - real_out.mean()
        loss = reg_loss + cdiv_loss

        # Logging
        self.log('loss', loss)
        self.log('loss_regularization', reg_loss)
        self.log('loss_contrastive_divergence', cdiv_loss)
        self.log('metrics_avg_real', real_out.mean())
        self.log('metrics_avg_fake', fake_out.mean())
        return loss

    def validation_step(self, batch, batch_idx):
        # For validating, we calculate the contrastive divergence between purely random images and unseen examples
        # Note that the validation/test step of energy-based models depends on what we are interested in the model
        real_imgs, _ = batch
        fake_imgs = torch.rand_like(real_imgs) * 2 - 1

        inp_imgs = torch.cat([real_imgs, fake_imgs], dim=0)
        real_out, fake_out = self.cnn(inp_imgs).chunk(2, dim=0)

        cdiv = fake_out.mean() - real_out.mean()
        self.log('val_contrastive_divergence', cdiv)
        self.log('val_fake_out', fake_out.mean())
        self.log('val_real_out', real_out.mean())

We do not implement a test step because energy-based generative models are usually not evaluated on a test set. The validation step however is used to get an idea of the difference between energy/likelihood of random images to unseen examples of the dataset. Alternative test steps would be to generate new images and evaluate how realistic they are based on FID or Inception score, or try to denoise images.

### 3.5 Callbacks

To monitor progress in PyTorch Lightning, we create callbacks:
1. `GenerateCallback`: After every $N$ epochs (usually $N=5$ to reduce output to TensorBoard), it takes a small batch of random images and perform many MCMC iterations until the model's generation converges. Compared to the training that used 60 iterations, we use 256 here because (1) we only have to do it once compared to the training that has to do it every iteration, and (2) we do not start from a buffer here, but from scratch. 
2. `SamplerCallback`: Logs some of the buffer images.
3. `OutlierCallback`: Monitors how the network scores random noise.

In [ ]:
class GenerateCallback(pl.Callback):

    def __init__(self, batch_size=8, vis_steps=8, num_steps=256, every_n_epochs=5):
        super().__init__()
        self.batch_size = batch_size         # Number of images to generate
        self.vis_steps = vis_steps           # Number of steps within generation to visualize
        self.num_steps = num_steps           # Number of steps to take during generation
        self.every_n_epochs = every_n_epochs # Only save those images every N epochs (otherwise tensorboard gets quite large)

    def on_epoch_end(self, trainer, pl_module):
        # Skip for all other epochs
        if trainer.current_epoch % self.every_n_epochs == 0:
            # Generate images
            imgs_per_step = self.generate_imgs(pl_module)
            # Plot and add to tensorboard
            for i in range(imgs_per_step.shape[1]):
                step_size = self.num_steps // self.vis_steps
                imgs_to_plot = imgs_per_step[step_size-1::step_size,i]
                grid = torchvision.utils.make_grid(imgs_to_plot, nrow=imgs_to_plot.shape[0], normalize=True, range=(-1,1))
                trainer.logger.experiment.add_image(f"generation_{i}", grid, global_step=trainer.current_epoch)

    def generate_imgs(self, pl_module):
        pl_module.eval()
        start_imgs = torch.rand((self.batch_size,) + pl_module.hparams["img_shape"]).to(pl_module.device)
        start_imgs = start_imgs * 2 - 1
        torch.set_grad_enabled(True)  # Tracking gradients for sampling necessary
        imgs_per_step = Sampler.generate_samples(pl_module.cnn, start_imgs, steps=self.num_steps, step_size=10, return_img_per_step=True)
        torch.set_grad_enabled(False)
        pl_module.train()
        return imgs_per_step

The second callback is called `SamplerCallback`, and simply adds a randomly picked subset of images in the sampling buffer to the TensorBoard. This helps to understand what images are currently shown to the model as "fake".

In [ ]:
class SamplerCallback(pl.Callback):

    def __init__(self, num_imgs=32, every_n_epochs=5):
        super().__init__()
        self.num_imgs = num_imgs             # Number of images to plot
        self.every_n_epochs = every_n_epochs # Only save those images every N epochs (otherwise tensorboard gets quite large)

    def on_epoch_end(self, trainer, pl_module):
        if trainer.current_epoch % self.every_n_epochs == 0:
            exmp_imgs = torch.cat(random.choices(pl_module.sampler.examples, k=self.num_imgs), dim=0)
            grid = torchvision.utils.make_grid(exmp_imgs, nrow=4, normalize=True, range=(-1,1))
            trainer.logger.experiment.add_image("sampler", grid, global_step=trainer.current_epoch)

Finally, our last callback is `OutlierCallback`. This callback evaluates the model by recording the (negative) energy assigned to random noise. While our training loss is almost constant across iterations, this score is likely showing the progress of the model to detect "outliers".

In [ ]:
class OutlierCallback(pl.Callback):

    def __init__(self, batch_size=1024):
        super().__init__()
        self.batch_size = batch_size

    def on_epoch_end(self, trainer, pl_module):
        with torch.no_grad():
            pl_module.eval()
            rand_imgs = torch.rand((self.batch_size,) + pl_module.hparams["img_shape"]).to(pl_module.device)
            rand_imgs = rand_imgs * 2 - 1.0
            rand_out = pl_module.cnn(rand_imgs).mean()
            pl_module.train()

        trainer.logger.experiment.add_scalar("rand_out", rand_out, global_step=trainer.current_epoch)

### 3.6 Running the model

Finally, we can add everything together to create our final training function. The function is very similar to any other PyTorch Lightning training function we have seen so far. However, there is the small difference of that we do not test the model on a test set because we will analyse the model afterward by checking its prediction and ability to perform outlier detection.  

In [ ]:
def train_model(**kwargs):
    # Create a PyTorch Lightning trainer with the generation callback
    trainer = pl.Trainer(default_root_dir=os.path.join(CHECKPOINT_PATH, "MNIST"),
                         accelerator='auto',
                         devices=1,
                         max_epochs=60,
                         gradient_clip_val=0.1,
                         callbacks=[ModelCheckpoint(save_weights_only=True, mode="min", monitor='val_contrastive_divergence'),
                                    GenerateCallback(every_n_epochs=5),
                                    SamplerCallback(every_n_epochs=5),
                                    OutlierCallback(),
                                    LearningRateMonitor("epoch")
                                   ])
    # Check whether pretrained model exists. If yes, load it and skip training
    pretrained_filename = os.path.join(CHECKPOINT_PATH, "MNIST.ckpt")
    if os.path.isfile(pretrained_filename):
        print("Found pretrained model, loading...")
        model = DeepEnergyModel.load_from_checkpoint(pretrained_filename)
    else:
        pl.seed_everything(42)
        model = DeepEnergyModel(**kwargs)
        trainer.fit(model, train_loader, test_loader)
        model = DeepEnergyModel.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
    # No testing as we are more interested in other properties
    return model

In [ ]:
model = train_model(img_shape=(1, 28, 28),
                    batch_size=train_loader.batch_size,
                    lr=1e-4,
                    beta1=0.0)

## 4. Analysis

In the last part of the notebook, we will try to take the trained energy-based generative model, and analyse its properties.

### 4.1 TensorBoard

The first thing we can look at is the TensorBoard generate during training. This can help us to understand the training dynamic even better, and shows potential issues. Let's load the TensorBoard below:

In [ ]:
# Import tensorboard
%load_ext tensorboard

In [ ]:
# Opens tensorboard in notebook. Adjust the path to your CHECKPOINT_PATH!
%tensorboard --logdir models/tensorboards/

<center width="100%"><img src="imgs/tensorboard_screenshot.png" width="1000px"></center>

We see that the contrastive divergence as well as the regularization converge quickly to 0. However, the training continues although the loss is always close to zero. This is because our "training" data changes with the model by sampling. The progress of training can be best measured by looking at the samples across iterations, and the score for random images that decreases constantly over time.

### 4.2 Image Generation

Another way of evaluating generative models is by sampling a few generated images. Generative models need to be good at generating realistic images as this truely shows that they have modeled the true data distribution. Thus, let's sample a few images of the model below:

In [ ]:
model.to(device)
pl.seed_everything(43)
callback = GenerateCallback(batch_size=4, vis_steps=8, num_steps=256)
imgs_per_step = callback.generate_imgs(model)
imgs_per_step = imgs_per_step.cpu()

The characteristic of sampling with energy-based models is that they require the iterative MCMC algorithm. To gain an insight in how the images change over iterations, we plot a few intermediate samples in the MCMC as well:

In [ ]:
for i in range(imgs_per_step.shape[1]):
    step_size = callback.num_steps // callback.vis_steps
    imgs_to_plot = imgs_per_step[step_size-1::step_size,i]
    imgs_to_plot = torch.cat([imgs_per_step[0:1,i],imgs_to_plot], dim=0)
    grid = torchvision.utils.make_grid(imgs_to_plot, nrow=imgs_to_plot.shape[0], normalize=True, range=(-1,1), pad_value=0.5, padding=2)
    grid = grid.permute(1, 2, 0)
    plt.figure(figsize=(8,8))
    plt.imshow(grid)
    plt.xlabel("Generation iteration")
    plt.xticks([(imgs_per_step.shape[-1]+2)*(0.5+j) for j in range(callback.vis_steps+1)],
               labels=[1] + list(range(step_size,imgs_per_step.shape[0]+1,step_size)))
    plt.yticks([])
    plt.show()

We see that although starting from noise in the very first step, the sampling algorithm obtains reasonable shapes after only 32 steps. Over the next 200 steps, the shapes become clearer and changed towards realistic digits. The specific samples can differ when you run the code on Colab, hence the following description is specific to the plots shown on the website. The first row shows an 8, where we remove unnecessary white parts over iterations. The transformation across iterations can be seen at best for the second sample, which creates a digit of 2. While the first sample after 32 iterations looks a bit like a digit, but not really, the sample is transformed more and more to a typical image of the digit 2.

### 4.3 Out-of-distribution detection

A very common and strong application of energy-based models is out-of-distribution detection (sometimes referred to as "anomaly" detection). As more and more deep learning models are applied in production and applications, a crucial aspect of these models is to know what the models don't know. Deep learning models are usually overconfident, meaning that they classify even random images sometimes with 100% probability. Clearly, this is not something that we want to see in applications. Energy-based models can help with this problem because they are trained to detect images that do not fit the training dataset distribution. Thus, in those applications, you could train an energy-based model along with the classifier, and only output predictions if the energy-based models assign a (unnormalized) probability higher than $\delta$ to the image. You can actually combine classifiers and energy-based objectives in a single model, as proposed in this [paper](https://arxiv.org/abs/1912.03263).

In this part of the analysis, we want to test the out-of-distribution capability of our energy-based model. Remember that a lower output of the model denotes a low probability. Thus, we hope to see low scores if we enter random noise to the model:

In [ ]:
with torch.no_grad():
    rand_imgs = torch.rand((128,) + model.hparams.img_shape).to(model.device)
    rand_imgs = rand_imgs * 2 - 1.0
    rand_out = model.cnn(rand_imgs).mean()
    print(f"Average score for random images: {rand_out.item():4.2f}")

As we hoped, the model assigns very low probability to those noisy images. As another reference, let's look at predictions for a batch of images from the training set:

In [ ]:
with torch.no_grad():
    train_imgs,_ = next(iter(train_loader))
    train_imgs = train_imgs.to(model.device)
    train_out = model.cnn(train_imgs).mean()
    print(f"Average score for training images: {train_out.item():4.2f}")

The scores are close to 0 because of the regularization objective that was added to the training. So clearly, the model can distinguish between noise and real digits. However, what happens if we change the training images a little, and see which ones gets a very low score?

In [ ]:
@torch.no_grad()
def compare_images(img1, img2):
    imgs = torch.stack([img1, img2], dim=0).to(model.device)
    score1, score2 = model.cnn(imgs).cpu().chunk(2, dim=0)
    grid = torchvision.utils.make_grid([img1.cpu(), img2.cpu()], nrow=2, normalize=True, range=(-1,1), pad_value=0.5, padding=2)
    grid = grid.permute(1, 2, 0)
    plt.figure(figsize=(4,4))
    plt.imshow(grid)
    plt.xticks([(img1.shape[2]+2)*(0.5+j) for j in range(2)],
               labels=["Original image", "Transformed image"])
    plt.yticks([])
    plt.show()
    print(f"Score original image: {score1.item():4.2f}")
    print(f"Score transformed image: {score2.item():4.2f}")

We use a random test image for this. Feel free to change it to experiment with the model yourself.

In [ ]:
test_imgs, _ = next(iter(test_loader))
exmp_img = test_imgs[0].to(model.device)

The first transformation is to add some random noise to the image:

In [ ]:
img_noisy = exmp_img + torch.randn_like(exmp_img) * 0.3
img_noisy.clamp_(min=-1.0, max=1.0)
compare_images(exmp_img, img_noisy)

We can see that the score considerably drops. Hence, the model can detect random Gaussian noise on the image. This is also to expect as initially, the "fake" samples are pure noise images.

Next, we flip an image and check how this influences the score:

In [ ]:
img_flipped = exmp_img.flip(dims=(1,2))
compare_images(exmp_img, img_flipped)

If the digit can only be read in this way, for example, the 7, then we can see that the score drops. However, the score only drops slightly. This is likely because of the small size of our model. Keep in mind that generative modeling is a much harder task than classification, as we do not only need to distinguish between classes but learn **all** details/characteristics of the digits. With a deeper model, this could eventually be captured better (but at the cost of greater training instability).

Finally, we check what happens if we reduce the digit significantly in size:

In [ ]:
img_tiny = torch.zeros_like(exmp_img)-1
img_tiny[:,exmp_img.shape[1]//2:,exmp_img.shape[2]//2:] = exmp_img[:,::2,::2]
compare_images(exmp_img, img_tiny)

The score again drops but not by a large margin, although digits in the MNIST dataset usually are much larger.

Overall, we can conclude that our model is good for detecting Gaussian noise and smaller transformations to existing digits. Nonetheless, to obtain a very good out-of-distribution model, we would need to train deeper models and for more iterations.

### 4.4 Instability

Finally, we should discuss the possible instabilities of energy-based models, in particular for the example of image generation that we have implemented in this notebook. In the process of hyperparameter search for this notebook, there have been several models that diverged. Divergence in energy-based models means that the models assign a high probability to examples of the training set which is a good thing. However, at the same time, the sampling algorithm fails and only generates noise images that obtain minimal probability scores. This happens because the model has created many local maxima in which the generated noise images fall. The energy surface over which we calculate the gradients to reach data points with high probability has "diverged" and is not useful for our MCMC sampling.

Besides finding the optimal hyperparameters, a common trick in energy-based models is to reload stable checkpoints. If we detect that the model is diverging, we stop the training, load the model from one epoch ago where it did not diverge yet. Afterward, we continue training and hope that with a different seed the model is not diverging again. Nevertheless, this should be considered as the "last hope" for stabilizing the models, and careful hyperparameter tuning is the better way to do so. Sensitive hyperparameters include `step_size`, `steps` and the noise standard deviation in the sampler, and the learning rate and feature dimensionality in the CNN model.

## 5. Conclusion

In this notebook, we discussed concepts related to energy-based models.

We still see that training energy-based models is complex and can require frequent hyperparameter tuning. However, EBMs remain a compelling approach, unifying classification, outlier detection, and data generation under a single probabilistic framework.